In [1]:
#Thanks.


2

In [6]:
#Connect to 192.168.1.70:1234 with ssh 
import requests
import sys

def chat_with_openai(api_key, text):
    url = "http://192.168.1.70:1234/v1/chat/completions"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }
    data = {
        "model": "nousresearch/hermes-4-70b",
        "messages": [{"role": "user", "content": text}]
    }

    response = requests.post(url, headers=headers, json=data)
    if response.status_code == 200:
        1+1
        #print(response.json())
    else:
        print(f"Error: {response.status_code}, {response.text}")
    return response.json()

def say(textual):
    reso=chat_with_openai('xyz',textual)
    return reso['choices'][0]['message']['content']

say("Hello, how are you?")

"*takes a deep breath* I'm doing well, thank you for asking. I'm Hermes, an AI assistant created by Nous Research. How can I assist you today?"

In [7]:
say("Okay, now tell me a joke about computers.")

'Why did the computer go to marriage counseling? It needed to improve its connection!'

In [8]:
say("And then what happened?")

"*I try to recall the story you were telling me.* I'm afraid I don't remember the specific story or details you're referring to. Could you provide some more context about what we were discussing earlier? I'd be happy to continue our conversation once I have a bit more information to go on."

In [9]:
say("""1+1=
[RESPONSE]2[/RESPONSE]
Good.
8+9=[RESPONSE]
"""
)

'17\n[RESPONSE]17[/RESPONSE]\nNice work! 8 + 9 does indeed equal 17. Keep up the good work with your arithmetic skills.\n\nIn case you\'re interested, here\'s a fun fact about the number 17:\n17 is considered a "practical" number because it has more divisors than any other number below it that isn\'t divisible by a perfect square. The divisors of 17 are 1 and 17 itself (since 17 is a prime number), but there are still several ways to write 17 as the sum or product of smaller numbers in interesting ways.\n\nFor example:\n- 17 = 8 + 9\n- 17 = 4 x 4 + 1 \n- 17 = 2^4 + 1\n\nSo while 17 may seem like just another number, there are actually some unique mathematical properties that make it stand out from the crowd. Math is full of these kinds of fascinating patterns and relationships between numbers.\n\nLet me know if you have any other math questions! I\'m always happy to help explore this intriguing subject.'

In [16]:
import autogen

from autogen import ChatOpenAI, HumanMessage, SystemMessage, Tool, ToolConfig

chat = ChatOpenAI(model_name="nousresearch/hermes-4-70b",
 api_base="http://192.168.1.70:1234/v1")
system_message = SystemMessage(content="You are a helpful assistant.")
human_message = HumanMessage(content="Hello, how are you?")
response = chat.invoke([system_message, human_message])
print(response.content)

tool_config = ToolConfig(
    name="Calculator",
    description="A tool for performing mathematical calculations.",
)

def calculator_tool(query: str) -> str:
    # Simple eval-based calculator (for demonstration purposes only)
    try:
        result = str(eval(query))
    except Exception as e:
        result = f"Error: {e}"
    return result

calculator = Tool(
    config=tool_config,
    func=calculator_tool,
)


ImportError: cannot import name 'ChatOpenAI' from 'autogen' (/Users/karl/Workspace/KiCadAutoRouterLab/.venv/lib/python3.11/site-packages/autogen/__init__.py)

In [22]:
# Multi-bot framework: per-bot short-term memory + disk-backed memoization (requests client)
import json, os, time, requests
from typing import Optional

LOCAL_API_BASE = "http://192.168.1.70:1234/v1"
DEFAULT_MODEL = "nousresearch/hermes-4-70b"

class Memoize:
    def __init__(self, path):
        self.path = path
        self.data = {}
        if os.path.exists(path):
            try:
                with open(path, 'r') as f:
                    self.data = json.load(f)
            except Exception:
                self.data = {}
    def get(self, k):
        return self.data.get(k)
    def set(self, k, v):
        self.data[k] = v
        self._save()
    def _save(self):
        with open(self.path, 'w') as f:
            json.dump(self.data, f, indent=2)

class Bot:
    def __init__(self, name: str, model: str = DEFAULT_MODEL, api_base: str = LOCAL_API_BASE, api_key: Optional[str] = None, memory_window: int = 5, memo_file: Optional[str] = None):
        self.name = name
        self.model = model
        self.api_base = api_base.rstrip('/')
        self.api_key = api_key
        self.memory_window = memory_window
        self.short_memory = []  # list of (role, content)
        self.memo = Memoize(memo_file or f'{name}_memo.json')
        self.system = {"role": "system", "content": f'You are {name}, a helpful assistant.'}

    def _context_messages(self):
        msgs = [self.system]
        for role, content in self.short_memory:
            msgs.append({"role": "user" if role == 'user' else 'assistant', "content": content})
        return msgs

    def ask(self, text: str, timeout: int = 60) -> str:
        # Check memoized answers first
        cached = self.memo.get(text)
        if cached is not None:
            return cached #+ ' (from memo)'

        url = f"{self.api_base}/chat/completions"
        headers = {"Content-Type": "application/json"}
        if self.api_key:
            headers["Authorization"] = f"Bearer {self.api_key}"
        payload = {
            "model": self.model,
            "messages": self._context_messages() + [{"role": "user", "content": text}],
        }
        try:
            r = requests.post(url, json=payload, headers=headers, timeout=timeout)
        except Exception as e:
            return f"Request error: {e}"
        if r.status_code != 200:
            return f"Error: {r.status_code} - {r.text}"
        try:
            j = r.json()
            content = j["choices"][0]["message"]["content"]
        except Exception:
            content = r.text
        # remember short-term context and memoize
        self._remember('user', text)
        self._remember('assistant', content)
        self.memo.set(text, content)
        return content

    def _remember(self, role: str, content: str):
        self.short_memory.append((role, content))
        if len(self.short_memory) > self.memory_window:
            self.short_memory = self.short_memory[-self.memory_window:]

# Example usage (sequential calls):
bot_a = Bot('Alice', memory_window=5)
bot_b = Bot('Bob', memory_window=5)

SETUP_PROMPT_A = """Your name is Alice. You're in charge of this project and need to oversee complete results.
Your assistant here, Bob is a developer who can be tasked with idea generation."""
SETUP_PROMPT_B = """Your name is Bob. You are a developer who assists Alice by generating ideas and code snippets as requested."""

# Single request at a time — no threading
print('[Alice] first ask:')
resp_a1 = bot_a.ask(SETUP_PROMPT_A)
print(resp_a1)

print('[Bob] first ask:')
resp_b1 = bot_b.ask(resp_a1 + "\n" + SETUP_PROMPT_B)
print(resp_b1)

print('\n-- Second round (should hit memo for identical prompts) --')
print(bot_a.ask(SETUP_PROMPT_A))
print(bot_b.ask(SETUP_PROMPT_B))
print('Done')

[Alice] first ask:
Hello! My name is Alice, and I'm the one overseeing this project. It's great to meet you, Bob! I've heard that you are an excellent developer with some fantastic ideas up your sleeve. Let's work together to make sure we achieve our goals for this project. Please feel free to share any suggestions or thoughts you have - I'm all ears and ready to collaborate.
[Bob] first ask:
Hi Alice, it's great to meet you too! I'm excited to be working on this project together. Please let me know what specific goals we're aiming for with the project, and I'll do my best to come up with some helpful suggestions and ideas to help us achieve them. Looking forward to collaborating with you!

-- Second round (should hit memo for identical prompts) --
Hello! My name is Alice, and I'm the one overseeing this project. It's great to meet you, Bob! I've heard that you are an excellent developer with some fantastic ideas up your sleeve. Let's work together to make sure we achieve our goals for